# Restaurant Review Sentiment Analysis

Classify English restaurant reviews as positive (`1`) or negative (`0`) using Bag-of-Words features, logistic regression, and random forest. See [README.md](README.md) for setup and [preparation notes](docs/preparation-notes.md) for changes to the supplied notebook. Run from the repository folder.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 0
ARTIFACT_DIR = Path("artifacts")


In [ ]:
DATA_PATH = Path("Restaurant_Reviews.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError("Run this notebook from the project folder containing Restaurant_Reviews.csv.")
df = pd.read_csv(DATA_PATH)
if df.columns.tolist() != ["Review", "Liked"]:
    raise ValueError("Expected columns: Review, Liked")
if df.isna().any().any() or not df["Liked"].isin([0, 1]).all():
    raise ValueError("Dataset must contain non-missing reviews and binary 0/1 labels.")
if not df["Review"].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
    raise ValueError("Review must contain non-empty text.")
if (df.groupby("Review")["Liked"].nunique() > 1).any():
    raise ValueError("Identical review text has conflicting labels.")
original_count = len(df)
df = df.drop_duplicates(subset=["Review"]).reset_index(drop=True)
print(f"Loaded {original_count} reviews; removed {original_count - len(df)} exact duplicates; using {len(df)}.")


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['Liked'].value_counts()

In [ ]:
df.isna().sum()

In [ ]:
df.head()

In [ ]:
df['Review'][0]

In [ ]:
import re
review = re.sub( '[^a-zA-Z]' , ' ' , df['Review'][0] )   #replaces not an (upper and lower case) character with space
print(review)

In [ ]:
re.sub( '[^a-zA-Z]' , ' ' , "Hi @Boston, I 123am #India......Love my country!!!!" )

In [ ]:
review = review.lower()
print(review)

In [ ]:
type(review)

In [ ]:
# Tokenization
review = review.split()
print(review)

In [ ]:
import nltk
from nltk.corpus import stopwords
try:
    stopwords.words("english")
except LookupError as exc:
    raise LookupError(
        "Install the NLTK stopwords data in this environment: "
        "python -m nltk.downloader stopwords"
    ) from exc
# Preserve negations, which can change sentiment.
STOP_WORDS = set(stopwords.words("english")) - {"no", "nor", "not", "never"}


In [ ]:
stopwords.words('english')

In [ ]:
len(stopwords.words('english'))

In [ ]:
stopwords.words('german')

In [ ]:
stopwords.words('french')

In [ ]:
review

In [ ]:
# Remove stopwords while retaining negation words.
review1 = [word for word in review if word not in STOP_WORDS]
print(review1)


In [ ]:
# Stemming
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
review1 = [ ps.stem(word) for word in review1 ]
print(review1)
# loved shortened to love

In [ ]:
review2 = ' '.join(review1)
print(review2)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Create an instance of CountVectorizer
vectorizer = CountVectorizer()
# try with max_features = 5

# Sample corpus of text documents
corpus = [
    "I love to eat pizza",
    "Pizza is my favorite food",
    "I eat pizza every day"
]

# Fit the vectorizer to the corpus of text documents
vectorizer.fit(corpus)

# Get the vocabulary (unique words) from the corpus
vocabulary = vectorizer.vocabulary_
vocabulary_with_keys = list(vectorizer.vocabulary_.keys())

# Transform the corpus into a document-term matrix
X = vectorizer.transform(corpus)

# Print the document-term matrix
print(X.toarray())

# Print the vocabulary
print(vocabulary)
print(vocabulary_with_keys)
# "I love to eat pizza",
# "Pizza is my favorite food",
# "I eat pizza every day"


In [ ]:
# Bag of Words: CountVectorizer()
# used for converting a collection of text documents into a matrix of token counts
corpus1 = []
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
print(review2)
corpus1.append(review2)
print(corpus1)
X = cv.fit_transform(corpus1)

In [ ]:
# all three words are present in the vocabulory
X.toarray()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
import re
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def preprocess_review(text):
    """Apply the same deterministic cleaning to training and prediction inputs."""
    if not isinstance(text, str):
        raise TypeError("Review must be a string.")
    text = text.lower().replace("’", "'")
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"n't\b", " not", text)
    tokens = re.sub("[^a-zA-Z]", " ", text).split()
    return " ".join(ps.stem(word) for word in tokens if word not in STOP_WORDS)

corpus = [preprocess_review(text) for text in df["Review"]]
print("Cleaned reviews:", len(corpus))
print("Empty cleaned reviews:", sum(not text for text in corpus))


In [ ]:
corpus[:10]


In [ ]:
corpus_dataset = pd.DataFrame(corpus)
corpus_dataset.head()

In [ ]:
corpus_dataset.columns=['corpus']

In [ ]:
corpus_dataset.head()

In [ ]:
ARTIFACT_DIR.mkdir(exist_ok=True)
corpus_dataset.to_csv(ARTIFACT_DIR / "cleaned_reviews.csv", index=False)


In [ ]:
type(corpus_dataset)

In [ ]:
type(corpus)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# The vocabulary is fitted only after the train/test split below.
cv = CountVectorizer(max_features=1500)


In [ ]:
# Keep cleaned strings until the split; do not learn a vocabulary from all reviews.
X = np.asarray(corpus, dtype=object)


In [ ]:
X[:5]


In [ ]:
X[0]

In [ ]:
# Number of cleaned tokens in the first example.
len(X[0].split())


In [ ]:
len(X)

In [ ]:
X.shape

In [ ]:
# Extract the target by its explicit column name.
y = df["Liked"].to_numpy()


In [ ]:
y.shape

In [ ]:
np.unique(y, return_counts=True)


In [ ]:
df.head()

### Model training

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
# Keep identical cleaned text in one partition, including variants that
# become identical after punctuation removal, stopword removal, or stemming.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_indices, test_indices = next(splitter.split(X, y, groups=X))
text_train, text_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]
assert set(text_train).isdisjoint(text_test)
X_train = cv.fit_transform(text_train)
X_test = cv.transform(text_test)
print("Training:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
logistic_model = LogisticRegression(max_iter=1000, random_state=SEED)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
print("Logistic regression accuracy:", accuracy_score(y_test, logistic_predictions))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest_model = RandomForestClassifier(n_estimators=100, random_state=SEED)
forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)
print("Random forest accuracy:", accuracy_score(y_test, forest_predictions))


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)
models = {"Logistic regression": logistic_model, "Random forest": forest_model}


In [ ]:
results = []
confusion_matrices = {}
for name, fitted_model in models.items():
    predictions = fitted_model.predict(X_test)
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    })
    confusion_matrices[name] = confusion_matrix(y_test, predictions, labels=[0, 1]).tolist()
    print(name)
    print(classification_report(y_test, predictions, labels=[0, 1],
                                target_names=["Negative", "Positive"], zero_division=0))
results_df = pd.DataFrame(results).set_index("model")
display(results_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, matrix) in zip(axes, confusion_matrices.items()):
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
    ax.set(title=name, xlabel="Predicted", ylabel="Actual")
plt.tight_layout()
plt.show()


In [ ]:
# Use the same cleaner and vocabulary as in training for new reviews.
example_reviews = ["not nice service", "not a great taste", "The food was delicious and the staff were friendly."]
cleaned_examples = [preprocess_review(text) for text in example_reviews]
example_features = cv.transform(cleaned_examples)
example_predictions = pd.DataFrame({"Review": example_reviews, "Cleaned": cleaned_examples})
for name, fitted_model in models.items():
    labels = fitted_model.predict(example_features)
    example_predictions[name] = np.where(labels == 1, "Positive", "Negative")
display(example_predictions)


In [ ]:
# Change this review and run the cell again after training.
user_review = "not a great taste"
cleaned_review = preprocess_review(user_review)
if not cleaned_review:
    raise ValueError("This review has no usable tokens after preprocessing.")
features = cv.transform([cleaned_review])
if features.nnz == 0:
    raise ValueError("No words from this review occur in the training vocabulary.")
for name, fitted_model in models.items():
    label = int(fitted_model.predict(features)[0])
    print(f"{name}: {'Positive' if label == 1 else 'Negative'}")
